In [ ]:
import importlib.util, subprocess, sys

pkgs = {
    'transformers': 'transformers>=4.40.0',
    'torchvision': 'torchvision',
    'sklearn': 'scikit-learn',
    'cv2': 'opencv-python-headless',
    'PIL': 'pillow',
    'tqdm': 'tqdm',
    'huggingface_hub': 'huggingface_hub',
}
missing = [pkg for mod, pkg in pkgs.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
try:
    from transformers import Sam3Processor, Sam3Model
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers'])
    try:
        from transformers import Sam3Processor, Sam3Model
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'git+https://github.com/huggingface/transformers'])
        from transformers import Sam3Processor, Sam3Model
print('[setup] ready')


In [ ]:
from pathlib import Path
from collections import defaultdict
import gc, json, math, os, random

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.metrics import adjusted_rand_score

import torch
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import login
from transformers import AutoImageProcessor, AutoModel, Sam3Processor, Sam3Model

ImageFile.LOAD_TRUNCATED_IMAGES = True
SEED = 20260424
VERSION = 'sam3_dino_patch_graph_v20260424'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = WORK_DIR / 'animalclef_sam3_dino_patch'
CROP_DIR = OUT_DIR / 'masked_crops'
CACHE_DIR = OUT_DIR / 'cache'
REPORT_DIR = OUT_DIR / 'reports'
SUB_DIR = OUT_DIR / 'submissions'
for d in [OUT_DIR, CROP_DIR, CACHE_DIR, REPORT_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))

token = os.environ.get('HF_TOKEN')
if token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        token = None
if token:
    login(token=token, add_to_git_credential=False)

def find_competition_root():
    candidates = [Path('/kaggle/input/animal-clef-2026'), Path('/kaggle/input/competitions/animal-clef-2026'), Path('/content/animal-clef-2026'), Path.cwd() / 'animal-clef-2026']
    for root in candidates:
        if (root / 'metadata.csv').exists() and (root / 'sample_submission.csv').exists():
            return root
    for base in [Path('/kaggle/input'), Path('/content'), Path.cwd()]:
        if not base.exists():
            continue
        for p in base.rglob('metadata.csv'):
            if (p.parent / 'sample_submission.csv').exists():
                return p.parent
    raise FileNotFoundError('competition data not found')

DATA_ROOT = find_competition_root()
meta = pd.read_csv(DATA_ROOT / 'metadata.csv').reset_index(drop=True)
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
meta['row_idx'] = np.arange(len(meta))
meta['orientation'] = meta.get('orientation', '').fillna('').astype(str)
meta['date_dt'] = pd.to_datetime(meta.get('date'), errors='coerce')
meta['abs_path'] = meta['path'].map(lambda p: str((DATA_ROOT / str(p)).resolve()) if not Path(str(p)).is_absolute() else str(p))
print(meta.groupby(['dataset', 'split']).size())


In [ ]:
# Controls.
SAM3_MAX_SIDE = 896
SAM3_BATCH_SIZE = 8
BACKGROUND_COLOR = (238, 238, 232)

PROMPT = {
    'LynxID2025': 'lynx animal',
    'SalamanderID2025': 'salamander animal',
    'SeaTurtleID2022': 'sea turtle head animal',
    'TexasHornedLizards': 'horned lizard animal',
}
SAM3_THRESHOLD = {'LynxID2025': 0.35, 'SalamanderID2025': 0.30, 'SeaTurtleID2022': 0.35, 'TexasHornedLizards': 0.30}
SAM3_MASK_THRESHOLD = 0.50

DINO_MODEL = 'facebook/dinov2-base'
DINO_SIZE = 224
DINO_BATCH_SIZE = 48
PATCH_KEEP = 96
PATCH_TOPM = 16

TOPK_GRID = [10, 20, 30]
EDGE_GRID = np.round(np.linspace(0.45, 0.82, 13), 3).tolist()
KNOWN_GRID = [0.78, 0.84, 0.90, 0.96]
MARGIN_THR = 0.04
FINAL_TOPK = {'LynxID2025': 20, 'SalamanderID2025': 20, 'SeaTurtleID2022': 20, 'TexasHornedLizards': 25}
FINAL_EDGE = {'LynxID2025': 0.64, 'SalamanderID2025': 0.66, 'SeaTurtleID2022': 0.63, 'TexasHornedLizards': 0.66}
FINAL_KNOWN = {'LynxID2025': 0.90, 'SalamanderID2025': 0.96, 'SeaTurtleID2022': 0.90, 'TexasHornedLizards': 1.10}
MAX_COMPONENT = {'LynxID2025': 70, 'SalamanderID2025': 25, 'SeaTurtleID2022': 35, 'TexasHornedLizards': 30}
ANCHOR_BUCKET_CAP = {'LynxID2025': 24, 'SalamanderID2025': 12, 'SeaTurtleID2022': 18, 'TexasHornedLizards': 0}
PATCH_PAIR_CAP = {'LynxID2025': 4000, 'SalamanderID2025': 3500, 'SeaTurtleID2022': 2500, 'TexasHornedLizards': 1800}
print('version:', VERSION)


In [ ]:
sam_model = Sam3Model.from_pretrained('facebook/sam3').to(DEVICE).eval()
sam_processor = Sam3Processor.from_pretrained('facebook/sam3')

def load_resized_rgb(path, max_side=SAM3_MAX_SIDE):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return img

def choose_mask(masks, scores):
    if masks is None or len(masks) == 0:
        return None
    arr = masks.detach().cpu().numpy().astype(bool)
    scores_np = scores.detach().cpu().numpy() if scores is not None else np.ones(len(arr), dtype='float32')
    h, w = arr.shape[-2:]
    total_area = float(h * w)
    best = None
    best_score = -1.0
    for mask, score in zip(arr, scores_np):
        area = float(mask.sum())
        frac = area / total_area
        if frac < 0.004 or frac > 0.92:
            continue
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue
        cx = (xs.mean() / max(w - 1, 1)) - 0.5
        cy = (ys.mean() / max(h - 1, 1)) - 0.5
        centrality = 1.0 - min(1.0, math.sqrt(cx * cx + cy * cy) / 0.70)
        rank = float(score) * 0.68 + min(frac / 0.33, 1.0) * 0.18 + centrality * 0.14
        if rank > best_score:
            best_score = rank
            best = mask
    return best

def save_masked_crop(path, species, image, mask):
    path = Path(path)
    out_dir = CROP_DIR / species
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (path.stem + '_masked.jpg')
    if out_path.exists():
        return str(out_path), True
    img = np.array(image)
    if mask is None:
        crop = img
        ok = False
    else:
        ys, xs = np.where(mask)
        if len(xs) == 0:
            crop = img
            ok = False
        else:
            h, w = mask.shape
            margin = int(0.10 * max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)) + 10
            x0, x1 = max(0, xs.min() - margin), min(w, xs.max() + margin + 1)
            y0, y1 = max(0, ys.min() - margin), min(h, ys.max() + margin + 1)
            bg = np.zeros_like(img)
            bg[:, :] = np.array(BACKGROUND_COLOR, dtype=np.uint8)
            crop = np.where(mask[..., None], img, bg)[y0:y1, x0:x1]
            ok = True
    if min(crop.shape[:2]) < 64:
        crop = img
        ok = False
    cv2.imwrite(str(out_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    return str(out_path), ok

def segment_batch(paths, species, batch_size=SAM3_BATCH_SIZE):
    paths = [str(p) for p in paths]
    outputs = [None] * len(paths)
    pending_slots, pending_paths, pending_images = [], [], []
    for slot, path in enumerate(paths):
        p = Path(path)
        out_path = CROP_DIR / species / (p.stem + '_masked.jpg')
        if out_path.exists():
            outputs[slot] = (str(out_path), True)
        else:
            pending_slots.append(slot)
            pending_paths.append(p)
            pending_images.append(load_resized_rgb(p))
    prompt = PROMPT[species]
    for start in tqdm(range(0, len(pending_paths), batch_size), desc=f'SAM3 {species}'):
        batch_slots = pending_slots[start:start + batch_size]
        batch_paths = pending_paths[start:start + batch_size]
        batch_images = pending_images[start:start + batch_size]
        masks = [None] * len(batch_images)
        try:
            inputs = sam_processor(images=batch_images, text=[prompt] * len(batch_images), return_tensors='pt').to(DEVICE)
            with torch.inference_mode():
                if DEVICE == 'cuda':
                    with torch.autocast(device_type='cuda', dtype=torch.float16):
                        out = sam_model(**inputs)
                else:
                    out = sam_model(**inputs)
            results = sam_processor.post_process_instance_segmentation(out, threshold=SAM3_THRESHOLD[species], mask_threshold=SAM3_MASK_THRESHOLD, target_sizes=inputs.get('original_sizes').tolist())
            for k, result in enumerate(results):
                masks[k] = choose_mask(result.get('masks'), result.get('scores'))
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and DEVICE == 'cuda' and batch_size > 1:
                torch.cuda.empty_cache()
                retry = segment_batch([str(p) for p in batch_paths], species, batch_size=1)
                for slot, r in zip(batch_slots, retry):
                    outputs[slot] = r
                continue
            print('[sam3] batch failed:', repr(e))
        except Exception as e:
            print('[sam3] batch failed:', repr(e))
        for slot, p, img, mask in zip(batch_slots, batch_paths, batch_images, masks):
            outputs[slot] = save_masked_crop(p, species, img, mask)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    return outputs

crop_path_by_row = {}
for species, g in meta.groupby('dataset'):
    results = segment_batch(g['abs_path'].tolist(), species)
    print(f'[sam3] {species} masks ok {sum(ok for _, ok in results)}/{len(results)}')
    for row_idx, (crop, ok) in zip(g['row_idx'].tolist(), results):
        crop_path_by_row[int(row_idx)] = crop
meta['crop_path'] = meta['row_idx'].map(crop_path_by_row)
del sam_model
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()


In [ ]:
processor = AutoImageProcessor.from_pretrained(DINO_MODEL)
dino = AutoModel.from_pretrained(DINO_MODEL).to(DEVICE).eval()

class DinoCropDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (DINO_SIZE, DINO_SIZE), BACKGROUND_COLOR)
        img = img.resize((DINO_SIZE, DINO_SIZE), Image.Resampling.BICUBIC)
        return np.array(img)

def l2n(x):
    x = np.asarray(x, dtype=np.float32)
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def extract_dino(paths, prefix):
    cls_cache = CACHE_DIR / f'{prefix}_cls.npy'
    patch_cache = CACHE_DIR / f'{prefix}_patch.npy'
    if cls_cache.exists() and patch_cache.exists():
        cls = np.load(cls_cache)
        patch = np.load(patch_cache)
        if len(cls) == len(paths):
            print('[cache] loaded', prefix, cls.shape, patch.shape)
            return cls.astype('float32'), patch.astype('float32')
    ds = DinoCropDataset(paths)
    loader = DataLoader(ds, batch_size=DINO_BATCH_SIZE, shuffle=False, num_workers=2)
    cls_out, patch_out = [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc=f'dino {prefix}'):
            inputs = processor(images=list(batch.numpy()), return_tensors='pt').to(DEVICE)
            if DEVICE == 'cuda':
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out = dino(**inputs)
            else:
                out = dino(**inputs)
            hidden = out.last_hidden_state.float()
            cls = hidden[:, 0]
            patches = hidden[:, 1:]
            # Keep the most salient tokens to make local matching cheaper.
            norms = patches.norm(dim=-1)
            k = min(PATCH_KEEP, patches.shape[1])
            top_idx = norms.topk(k=k, dim=1).indices
            gather_idx = top_idx.unsqueeze(-1).expand(-1, -1, patches.shape[-1])
            patches = torch.gather(patches, 1, gather_idx)
            cls_out.append(torch.nn.functional.normalize(cls, dim=1).cpu().numpy())
            patch_out.append(torch.nn.functional.normalize(patches, dim=2).cpu().numpy())
    cls_arr = np.concatenate(cls_out, axis=0).astype('float16')
    patch_arr = np.concatenate(patch_out, axis=0).astype('float16')
    np.save(cls_cache, cls_arr)
    np.save(patch_cache, patch_arr)
    return cls_arr.astype('float32'), patch_arr.astype('float32')

def side_bucket(x):
    s = str(x).lower()
    if 'left' in s: return 'left'
    if 'right' in s: return 'right'
    return None

def orient_penalty(sim, q_meta, g_meta, species):
    if species not in {'LynxID2025', 'SeaTurtleID2022'}:
        return sim
    q_side = [side_bucket(x) for x in q_meta['orientation'].tolist()]
    g_side = np.array([side_bucket(x) for x in g_meta['orientation'].tolist()], dtype=object)
    pen = 0.06 if species == 'LynxID2025' else 0.04
    for i, s in enumerate(q_side):
        if s is None: continue
        mask = np.array([(t is not None and t != s) for t in g_side])
        sim[i, mask] -= pen
    return sim

def patch_scores_batch(patch_a, patch_b, batch=64):
    scores = []
    for start in range(0, len(patch_a), batch):
        a = torch.from_numpy(patch_a[start:start + batch]).to(DEVICE)
        b = torch.from_numpy(patch_b[start:start + batch]).to(DEVICE)
        sim = torch.bmm(a, b.transpose(1, 2))
        top1 = sim.max(dim=2).values.topk(k=min(PATCH_TOPM, sim.shape[1]), dim=1).values.mean(dim=1)
        top2 = sim.max(dim=1).values.topk(k=min(PATCH_TOPM, sim.shape[2]), dim=1).values.mean(dim=1)
        sc = 0.5 * (top1 + top2)
        scores.append(sc.float().cpu().numpy())
    return np.concatenate(scores, axis=0)

class DSU:
    def __init__(self, n): self.p=list(range(n)); self.sz=[1]*n
    def find(self, x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self, a, b, max_size=None):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return True
        if max_size is not None and self.sz[ra] + self.sz[rb] > max_size: return False
        if self.sz[ra] < self.sz[rb]: ra, rb = rb, ra
        self.p[rb] = ra; self.sz[ra] += self.sz[rb]
        return True

def open_split(df, species, fold=0):
    rng = np.random.default_rng(SEED + fold * 97 + sum(map(ord, species)))
    ids = np.array(sorted(df['identity'].dropna().unique()), dtype=object).copy()
    rng.shuffle(ids)
    novel = set(ids[:max(1, int(0.22 * len(ids)))])
    gallery, query = [], []
    for ident, g in df.groupby('identity'):
        rows = g.index.to_numpy().copy(); rng.shuffle(rows)
        if ident in novel:
            query.extend(rows.tolist())
        elif len(rows) >= 2:
            query.append(int(rows[0])); gallery.extend(rows[1:].tolist())
        else:
            gallery.extend(rows.tolist())
    return np.array(gallery, dtype=int), np.array(query, dtype=int)

def mutual_edges(global_sim, topk):
    sim = global_sim.copy()
    np.fill_diagonal(sim, -9)
    k = min(int(topk), max(1, sim.shape[0] - 1))
    nbr = np.argpartition(-sim, kth=np.arange(k), axis=1)[:, :k]
    nbr_sets = [set(r.tolist()) for r in nbr]
    edges = []
    for i in range(sim.shape[0]):
        for j in nbr[i]:
            j = int(j)
            if j <= i: continue
            if i in nbr_sets[j]: edges.append((float(sim[i, j]), i, j))
    edges.sort(reverse=True)
    return edges

def fuse_edges(edges, patch_tokens, edge_thr, cap):
    candidates = [e for e in edges if e[0] >= edge_thr - 0.08]
    candidates.sort(key=lambda e: abs(e[0] - edge_thr))
    candidates = candidates[:cap]
    if not candidates:
        return edges
    ia = np.array([i for _, i, _ in candidates], dtype=int)
    ib = np.array([j for _, _, j in candidates], dtype=int)
    ps = patch_scores_batch(patch_tokens[ia], patch_tokens[ib], batch=64)
    patch_map = {(int(i), int(j)): float(s) for (_, i, j), s in zip(candidates, ps)}
    fused = []
    for score, i, j in edges:
        p = patch_map.get((i, j))
        if p is None:
            fused_score = score
        else:
            fused_score = 0.58 * score + 0.42 * p
            if p > edge_thr + 0.03 and score > edge_thr - 0.10:
                fused_score = max(fused_score, edge_thr + 0.01)
        fused.append((float(fused_score), i, j))
    fused.sort(reverse=True)
    return fused

def predict_species(sp_df, cls_feat, patch_feat, species, topk, edge_thr, known_thr):
    q_meta = sp_df.reset_index(drop=True)
    global_sim = cls_feat @ cls_feat.T
    global_sim = orient_penalty(global_sim, q_meta, q_meta, species)
    edges = mutual_edges(global_sim, topk)
    edges = fuse_edges(edges, patch_feat, edge_thr, PATCH_PAIR_CAP[species])
    dsu = DSU(len(sp_df))
    for score, i, j in edges:
        if score < edge_thr: break
        dsu.union(i, j, max_size=MAX_COMPONENT[species])
    train_mask = q_meta['split'].eq('train').to_numpy()
    if train_mask.any() and known_thr < 1.0 and ANCHOR_BUCKET_CAP[species] > 0:
        g_idx = np.where(train_mask)[0]
        q_idx = np.where(~train_mask)[0]
        qg = cls_feat[q_idx] @ cls_feat[g_idx].T
        qg = orient_penalty(qg, q_meta.iloc[q_idx].reset_index(drop=True), q_meta.iloc[g_idx].reset_index(drop=True), species)
        order = np.argpartition(-qg, kth=np.arange(min(2, qg.shape[1])), axis=1)[:, :min(2, qg.shape[1])]
        glabs = q_meta.iloc[g_idx]['identity'].astype(str).to_numpy()
        buckets = defaultdict(list)
        for local_i, row_i in enumerate(q_idx):
            cand = order[local_i]
            cand = cand[np.argsort(-qg[local_i, cand])]
            top_score = float(qg[local_i, cand[0]])
            second = float(qg[local_i, cand[1]]) if len(cand) > 1 else -1.0
            if top_score >= known_thr and (top_score - second) >= MARGIN_THR:
                buckets[glabs[int(cand[0])]].append(int(row_i))
        pair_thr = max(edge_thr - 0.04, 0.52)
        for lab, inds in buckets.items():
            if len(inds) > ANCHOR_BUCKET_CAP[species]:
                continue
            for a in range(len(inds)):
                i = inds[a]
                for j in inds[a+1:]:
                    if global_sim[i, j] >= pair_thr:
                        dsu.union(i, j, max_size=MAX_COMPONENT[species])
    roots = {}
    pred = []
    for i in range(len(sp_df)):
        if q_meta.iloc[i]['split'] != 'test':
            pred.append(None)
            continue
        r = dsu.find(i)
        if r not in roots:
            roots[r] = len(roots)
        pred.append(roots[r])
    return pred, edges

parts = []
reports = []
for species in ['LynxID2025', 'SalamanderID2025', 'SeaTurtleID2022', 'TexasHornedLizards']:
    sp = meta[meta['dataset'].eq(species)].copy().reset_index(drop=True)
    cls_feat, patch_feat = extract_dino(sp['crop_path'].tolist(), f'{species}_{VERSION}')
    tuned = {'local_ari': 'N/A', 'topk': FINAL_TOPK[species], 'edge_thr': FINAL_EDGE[species], 'known_thr': FINAL_KNOWN[species]}
    if sp['split'].eq('train').any() and species != 'TexasHornedLizards':
        best = (-9.0, None)
        tr = sp[sp['split'].eq('train')].copy().reset_index(drop=True)
        tr_cls, tr_patch = extract_dino(tr['crop_path'].tolist(), f'{species}_{VERSION}_trainonly')
        for topk in TOPK_GRID:
            for edge_thr in EDGE_GRID:
                for known_thr in KNOWN_GRID:
                    aris = []
                    for fold in range(2):
                        g_idx, q_idx = open_split(tr, species, fold)
                        fold_df = tr.iloc[np.concatenate([g_idx, q_idx])].copy().reset_index(drop=True)
                        fold_df.loc[:len(g_idx)-1, 'split'] = 'train'
                        fold_df.loc[len(g_idx):, 'split'] = 'test'
                        fold_cls = np.concatenate([tr_cls[g_idx], tr_cls[q_idx]], axis=0)
                        fold_patch = np.concatenate([tr_patch[g_idx], tr_patch[q_idx]], axis=0)
                        pred, _ = predict_species(fold_df, fold_cls, fold_patch, species, topk, float(edge_thr), float(known_thr))
                        true = fold_df[fold_df['split'].eq('test')]['identity'].astype(str).to_numpy()
                        pred = np.array([p for p in pred if p is not None])
                        aris.append(adjusted_rand_score(true, pred))
                    score = float(np.mean(aris))
                    if score > best[0]:
                        best = (score, {'local_ari': score, 'topk': int(topk), 'edge_thr': float(edge_thr), 'known_thr': float(known_thr), 'fold_ari': [float(x) for x in aris]})
        tuned = best[1]
        print('[tuned]', species, tuned)
    pred_all, edges = predict_species(sp, cls_feat, patch_feat, species, tuned['topk'], tuned['edge_thr'], tuned['known_thr'])
    test_mask = sp['split'].eq('test').to_numpy()
    test_ids = sp.loc[test_mask, 'image_id'].to_numpy()
    test_pred = [f'cluster_{species}_{p}' for p in pred_all if p is not None]
    part = pd.DataFrame({'image_id': test_ids, 'cluster': test_pred})
    vc = part['cluster'].value_counts()
    report = {'species': species, 'n_images': int(len(part)), 'n_clusters': int(part['cluster'].nunique()), 'max_cluster_size': int(vc.max()), 'singletons': int((vc==1).sum()), 'tuned': tuned}
    print('[report]', report)
    parts.append(part)
    reports.append(report)
    del cls_feat, patch_feat
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

sub = pd.concat(parts, ignore_index=True)
sub = sample[['image_id']].merge(sub, on='image_id', how='left')
assert len(sub) == len(sample)
assert sub['cluster'].notna().all()
versioned = SUB_DIR / f'submission_{VERSION}.csv'
sub.to_csv(versioned, index=False)
sub.to_csv(WORK_DIR / 'submission.csv', index=False)
summary = {'version': VERSION, 'reports': reports, 'submission': str(versioned)}
(REPORT_DIR / f'run_summary_{VERSION}.json').write_text(json.dumps(summary, indent=2))
print('wrote', versioned)
print('wrote', WORK_DIR / 'submission.csv')
print(json.dumps(summary, indent=2)[:4000])


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
for candidate in [repo, *repo.parents]:
    if (candidate / "paper_submissions_manifest.csv").exists():
        repo = candidate
        break
sys.path.insert(0, str(repo / "src"))

from notebook_tools import verify_submission
verify_submission(Path.cwd() / "submission.csv", "sam3_dino_patch_graph", repo)